# PAI for all sites at all scan positions in Berchtesgaden and Bosland based on hinge angle

This notebook generates different vegetation structure values such as PAI at 50 m height from ground from rxp and rdbx files of any scan position.

## Load all the required modules

In [1]:
!sudo mount -t drvfs L: /mnt/l

[sudo] password for riegl: 
sudo: a password is required
^C


In [1]:
import os
import csv
import re
import numpy as np
import glob

import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import MaxNLocator

from pylidar_tls_canopy import riegl_io, plant_profile_2, grid

from pathlib import Path
import shutil
import timeit

import pandas as pd
import re

# Set up directories
os.chdir('/mnt/l/projects/weave/chapter02')
cwd = os.getcwd()
print("Current Working Directory:", cwd)

# Load site info
sites_df = pd.read_csv("data/riscan_project_paths.csv")
mapping_df_bgd = pd.read_csv("data/BGD_10m_grid_scanpos.csv")
mapping_df_bos = pd.read_csv("data/BOS_10m_grid_scanpos.csv")

# Function to extract scanpos number
def extract_scanpos_number(path):
    match = re.search(r"ScanPos(\d+)", path)
    if match:
        return int(match.group(1))
    return None

Current Working Directory: /mnt/l/projects/weave/chapter02


In [2]:
print(sites_df)
#sites_df = sites_df.iloc[9:]

print(sites_df)

   site_id region  year                                        riscan_path
0   BGD001    BGD  2023        /mnt/d/TLS/RiSCAN/2023-07-09_Eisberg.RiSCAN
1   BGD001    BGD  2024        /mnt/d/TLS/RiSCAN/2024-09-04_Eisberg.RiSCAN
2   BGD002    BGD  2023  /mnt/l/projects/weave/data/tls/riscan/2023/202...
3   BGD002    BGD  2024      /mnt/d/TLS/RiSCAN/2024-05-29_Eisgraben.RiSCAN
4   BGD003    BGD  2023  /mnt/l/projects/weave/data/tls/riscan/2023/202...
5   BGD003    BGD  2024        /mnt/d/TLS/RiSCAN/2024-05-27_Endstal.RiSCAN
6   BGD004    BGD  2023  /mnt/l/projects/weave/data/tls/riscan/2023/202...
7   BGD004    BGD  2024        /mnt/d/TLS/RiSCAN/2024-06-04_Ofental.RiSCAN
8   BOS001    BOS  2023  /mnt/l/projects/weave/data/tls/riscan/2023/202...
9   BOS001    BOS  2024  /mnt/l/projects/weave/data/tls/riscan/2024/202...
10  BOS002    BOS  2023  /mnt/l/projects/weave/data/tls/riscan/2023/202...
11  BOS002    BOS  2024  /mnt/l/projects/weave/data/tls/riscan/2024/202...
12  BOS003    BOS  2023  

In [3]:
import sys
print(sys.executable)

/home/riegl/miniforge3/envs/ptc-chapter02/bin/python


In [ ]:
# Main loop
for i, row in sites_df.iterrows():
    site_path = row['riscan_path']
    region = row['region']
    site_id = row['site_id']
    year = row['year']
    dat_folder = os.path.join(site_path, "MATRICES")

    print(f"\nProcessing {site_id} ({region}) - {year}")
    rdbx_files = {}
    rxp_files = {}

    # Collect RDBX and RXP files
    for dirpath, dirnames, filenames in os.walk(site_path):
        match = re.search(r"ScanPos(\d+)", dirpath)
        if match and int(match.group(1)) % 2 == 1:
            for f in filenames:
                name, ext = os.path.splitext(f)
                full_path = os.path.join(dirpath, f)
                if ext == ".rdbx":
                    rdbx_files[name] = full_path
                elif ext == ".rxp":
                    rxp_files[name] = full_path

    # Match by filename
    matches = []
    for name in rdbx_files:
        if name in rxp_files:
            rdbx_path = rdbx_files[name]
            rxp_path = rxp_files[name]
            match = re.search(r"ScanPos(\d+)", rdbx_path)
            scanpos_str = match.group(1).zfill(3) if match else "XXX"
            dat_path = os.path.join(dat_folder, f"ScanPos{scanpos_str}.DAT")
            matches.append((rdbx_path, rxp_path, dat_path))

    if not matches:
        print(f"No matches found for {site_id}. Skipping...")
        continue

    # Save matched file paths
    matched_df = pd.DataFrame(matches, columns=['rdb_v', 'rxp_v', 'dat_v'])

    # Add scanpos_num
    matched_df['scanpos_num'] = matched_df['rxp_v'].apply(extract_scanpos_number)

    matched_df = matched_df.dropna(subset=['scanpos_num'])
    matched_df['scanpos_num'] = matched_df['scanpos_num'].astype(int)

    # Filter mapping for the current site
    # Select appropriate mapping based on region
    
    if region == 'BGD':
        current_mapping = mapping_df_bgd
    
    elif region == 'BOS':
        current_mapping = mapping_df_bos
    
    else:
        print(f"⚠️ Unknown region '{region}' for site {site_id}, skipping...")
        continue
    
    # Filter mapping for current site
    site_mapping = current_mapping[current_mapping['site'] == site_id]
    site_mapping = site_mapping.dropna(subset=[f"{year}_ScanPos_vert"])
    print(site_mapping)
    
    # Build scanpos-to-virtual_id mapping
    pos_to_id = dict(zip(site_mapping[f"{year}_ScanPos_vert"], site_mapping['virtual_id']))
    print(pos_to_id)
    print(matched_df)
    matched_df['scan_id'] = matched_df['scanpos_num'].map(pos_to_id)

    # Save intermediate file
    output_csv = f"data/matched_paths_{site_id}_{year}.csv"
    matched_df.to_csv(output_csv, index=False)
    print(f"[{site_id} {year}] Saved matched file list: {output_csv}")

    # Run your PAI computation for each scan
    def get_pgap_pai(scans):
        vert_rdbx_fn = scans['rdb_v']
        vert_rxp_fn = scans['rxp_v']
        vert_transform_fn = scans['dat_v']
        scan_id = scans['scan_id']

        if pd.isna(scan_id):
            print(f"Warning: No scan_id for {vert_rxp_fn}")
            return None

        # Load transformation matrix
        transform_matrix = riegl_io.read_transform_file(vert_transform_fn)
        x0, y0, z0, _ = transform_matrix[3, :]

        # Calculate PAI
        x, y, z, r = plant_profile_2.get_min_z_grid([vert_rdbx_fn], [vert_transform_fn], 
                                                  grid_extent=60, grid_resolution=1, 
                                                  grid_origin=[x0, y0], rxp=False)
        planefit = plant_profile_2.plane_fit_hubers(x, y, z, w=1/r)
        vpp_pai = plant_profile_2.Jupp2009(hres=0.5, zres=5, ares=90, 
                                         min_z=35, max_z=70, min_h=0, max_h=50,
                                         ground_plane=planefit['Parameters'])
        query_str = ['reflectance > -20']
        vpp_pai.add_riegl_scan_position(vert_rxp_fn, vert_transform_fn, sensor_height=1.8,
                                        rdbx_file=vert_rdbx_fn, method='WEIGHTED',
                                        min_zenith=35, max_zenith=70,
                                        query_str=query_str)
        vpp_pai.get_pgap_theta_z(min_azimuth=0, max_azimuth=360)
        hinge_pai = vpp_pai.calcHingePlantProfiles()
        pai_df = pd.DataFrame(hinge_pai)

        outdir = f"data/pai_hinge_vertical_new/{site_id}_{year}"
        os.makedirs(outdir, exist_ok=True)
        outname = f"{outdir}/pai_scan_{site_id}_{scan_id}_{year}.csv"
        pai_df.to_csv(outname, index=False)
        print(f"Saved PAI data: {outname}")
        return hinge_pai

    # Apply per-scan
    results = matched_df.apply(get_pgap_pai, axis=1)

# Done
print("\n✅ All sites processed.")


Processing BGD001 (BGD) - 2023
     fid    site virtual_id tm_id       height locality_id  2023_ScanPos_vert  \
0      1  BGD001        A01   NaN  1668.964111  BGD001_A01                1.0   
1      2  BGD001        A02   NaN  1673.062134  BGD001_A02               43.0   
2      3  BGD001        A03   NaN  1677.692139  BGD001_A03               45.0   
3      4  BGD001        A04   NaN  1682.240112  BGD001_A04               87.0   
4      5  BGD001        A05   NaN  1687.521118  BGD001_A05               89.0   
..   ...     ...        ...   ...          ...         ...                ...   
113  114  BGD001        K04   NaN  1727.526123  BGD001_K04               67.0   
114  115  BGD001        K05   NaN  1731.066040  BGD001_K05              109.0   
116  117  BGD001        K07   NaN  1737.680054  BGD001_K07              111.0   
118  119  BGD001        K09   NaN  1744.808105  BGD001_K09              133.0   
120  121  BGD001        K11   NaN  1749.001099  BGD001_K11              135.0